# Phần 1. NumPy trong workflow ML/DL

Các bài dưới đây dùng dữ liệu nhỏ để mô phỏng preprocessing, inference và xử lý
tensor trong một pipeline thực tế.

In [ ]:
STUDENT_NAME = "Tran Tan Thinh"  # TODO: Họ và tên
STUDENT_ID = "2510236"    # TODO: MSSV

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.8)

DATA_CANDIDATES = [
    Path("week02/numpy-pandas-eda-hw/data/automobile_raw.csv"),
    Path("data/automobile_raw.csv"),
    Path("../data/automobile_raw.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy data/automobile_raw.csv")

print("Data path:", DATA_PATH.resolve())

## N1. Stable softmax cho batch logits

Một classifier trả về `logits` có shape `(batch_size, num_classes)`. Tính softmax
theo từng mẫu bằng cách trừ giá trị lớn nhất trên mỗi hàng trước khi gọi `np.exp`.
Cách viết này tránh overflow khi logits có giá trị lớn.

**Biến đầu ra bắt buộc**

- `shifted_logits`: logits sau khi trừ row-wise maximum.
- `class_probabilities`: xác suất mỗi class, mỗi hàng có tổng bằng 1.
- `predicted_classes`: class có xác suất lớn nhất của từng mẫu.
- `confidence_scores`: xác suất lớn nhất của từng mẫu.

In [ ]:
logits = np.array([
    [2.0, 1.0, 0.1],
    [1000.0, 1001.0, 999.0],
    [-2.0, -1.0, 3.0],
    [0.5, 0.5, 0.5],
], dtype=np.float64)

In [ ]:
# TODO N1
shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
exp_logits = np.exp(shifted_logits)
class_probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
predicted_classes = np.argmax(class_probabilities, axis=1)
confidence_scores = np.max(class_probabilities, axis=1)

In [ ]:
required = [
    "shifted_logits",
    "class_probabilities",
    "predicted_classes",
    "confidence_scores",
]
if not all(name in globals() for name in required):
    print("Complete N1 to run this self-check.")
else:
    assert class_probabilities.shape == logits.shape
    assert np.all(np.isfinite(class_probabilities))
    assert np.allclose(class_probabilities.sum(axis=1), 1.0)
    assert predicted_classes.shape == (logits.shape[0],)
    assert confidence_scores.shape == (logits.shape[0],)
    print("N1 self-check passed")

## N2. Chuẩn hóa train và validation không gây leakage

Mỗi hàng là một mẫu, mỗi cột là một feature. Tính mean/std **chỉ từ `X_train`**,
sau đó dùng cùng thống kê để transform cả train và validation.

**Biến đầu ra bắt buộc**

- `train_feature_mean`, `train_feature_std`: shape `(4,)`.
- `X_train_scaled`: train set đã chuẩn hóa.
- `X_val_scaled`: validation set dùng thống kê từ train.

In [ ]:
# Features: height_cm, weight_kg, activity_hours, age
X_train = np.array([
    [170.0, 65.0, 1.2, 22.0],
    [180.0, 80.0, 2.4, 35.0],
    [160.0, 50.0, 0.8, 19.0],
    [175.0, 70.0, 1.5, 28.0],
    [168.0, 60.0, 1.0, 24.0],
    [182.0, 90.0, 3.0, 41.0],
])

X_val = np.array([
    [172.0, 68.0, 1.4, 26.0],
    [190.0, 95.0, 3.4, 45.0],
])

In [ ]:
# TODO N2
train_feature_mean = np.mean(X_train, axis=0)
train_feature_std = np.std(X_train, axis=0)
X_train_scaled = (X_train - train_feature_mean) / train_feature_std
X_val_scaled = (X_val - train_feature_mean) / train_feature_std

In [ ]:
required = [
    "train_feature_mean",
    "train_feature_std",
    "X_train_scaled",
    "X_val_scaled",
]
if not all(name in globals() for name in required):
    print("Complete N2 to run this self-check.")
else:
    assert X_train_scaled.shape == X_train.shape
    assert X_val_scaled.shape == X_val.shape
    assert np.allclose(X_train_scaled.mean(axis=0), 0.0)
    assert np.allclose(X_train_scaled.std(axis=0), 1.0)
    print("N2 self-check passed")

## N3. Tạo review queue sau inference

Giả sử `class_probabilities` đến từ N1. Một prediction cần được kiểm tra thủ công
nếu dự đoán sai **hoặc** confidence nhỏ hơn `0.70`.

**Biến đầu ra bắt buộc**

- `correct_mask`
- `high_confidence_mask`
- `review_mask`
- `review_indices`

In [ ]:
true_labels = np.array([0, 2, 2, 1])
confidence_threshold = 0.70

In [ ]:
# TODO N3
correct_mask = predicted_classes == true_labels
high_confidence_mask = confidence_scores >= confidence_threshold
# Đưa vào hàng chờ kiểm tra nếu dự đoán SAI hoặc CONFIDENCE THẤP
review_mask = ~(correct_mask & high_confidence_mask)
review_indices = np.where(review_mask)[0]

## N4. Tiền xử lý và augment một batch ảnh

`image_batch_uint8` có layout `(B, H, W, C)`. Chuyển batch về `float32` trong đoạn
`[0, 1]`, sau đó tạo một batch mới được flip ngang. Batch augment phải có bộ nhớ
độc lập để việc chỉnh sửa không làm thay đổi batch đã normalize.

Sau khi tạo batch augment, đặt pixel `augmented_batch[0, 0, 0, 0] = 1.0`.

**Biến đầu ra bắt buộc:** `normalized_batch`, `augmented_batch`.

In [ ]:
image_batch_uint8 = (
    np.arange(2 * 4 * 4 * 3, dtype=np.uint8)
    .reshape(2, 4, 4, 3)
)

In [ ]:
# TODO N4
normalized_batch = image_batch_uint8.astype(np.float32) / 255.0
augmented_batch = np.copy(np.flip(normalized_batch, axis=2))
augmented_batch[0, 0, 0, 0] = 1.0

# Phần 2. EDA với Automobile

Đọc `data/data_dictionary.md` trước khi xử lý.

## Câu hỏi mở đầu

1. Mỗi dòng đại diện cho đối tượng gì?
2. Ký hiệu missing value trong CSV là gì?
3. `symboling` có ý nghĩa gì?

**Trả lời**

<!-- Viết câu trả lời tại đây. -->
Dựa vào Data Dictionary được cung cấp trong file bài tập:
* Mỗi dòng đại diện cho một mẫu xe trong bộ 1985 Auto Imports.
* Ký hiệu missing value (giá trị bị thiếu) trong CSV là dấu ?.
* *symboling* thể hiện mức đánh giá rủi ro bảo hiểm (từ -3 đến 3).

## D1. Load và inspect raw CSV

Load dữ liệu sao cho dấu `?` vẫn là chuỗi để quan sát ảnh hưởng tới dtype.

**Biến đầu ra bắt buộc**

- `raw_df`: DataFrame raw.
- `raw_shape`: tuple.
- `raw_missing_marker_count`: tổng số dấu `?`.

In [ ]:
# TODO D1
raw_df = pd.read_csv(DATA_PATH)
raw_shape = raw_df.shape
raw_missing_marker_count = (raw_df == '?').sum().sum()

## D2. Missing values và dtype

1. Thay `?` bằng `np.nan`.
2. Chuyển các cột trong `NUMERIC_COLUMNS` bằng `pd.to_numeric`.
3. Tạo báo cáo missing.

**Biến đầu ra bắt buộc:** `df_clean`, `missing_by_column`.

In [ ]:
NUMERIC_COLUMNS = ['symboling', 'normalized_losses', 'wheel_base', 'length', 'width', 'height', 'curb_weight', 'engine_size', 'bore', 'stroke', 'compression_ratio', 'horsepower', 'peak_rpm', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D2
df_clean = raw_df.replace('?', np.nan)
for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(df_clean[column])
missing_by_column = df_clean.isna().sum()

### Giải thích cách làm sạch dữ liệu

- Vì sao không nên fill tất cả numeric columns bằng cùng một giá trị?
- Với `price`, lựa chọn drop hay fill phù hợp hơn cho bài EDA này? Vì sao?
- `normalized_losses` thiếu nhiều dữ liệu hơn các cột khác. Điều này ảnh hưởng thế nào?

**Nhận xét**

<!-- Viết 3--6 câu tại đây. -->
* Không nên fill tất cả numeric bằng một giá trị (VD: 0 hoặc mean trung tâm): Vì mỗi cột có phân phối, đơn vị và ý nghĩa khác nhau. Điền bừa bãi sẽ làm thay đổi độ lệch chuẩn và phá hỏng các mối tương quan (correlation).
* Với price: Trong bài toán EDA hoặc dự đoán giá, thường sẽ chọn drop (loại bỏ) các hàng bị thiếu giá. Vì price là biến mục tiêu (target variable), nếu ta fill nó bằng giá trị giả (như mean), ta đang đưa "nhiễu" vào dữ liệu và đánh lừa kết quả phân tích.
* normalized_losses thiếu nhiều: Có tới 41 giá trị thiếu. Việc loại bỏ hoàn toàn số hàng này (khoảng 20% dữ liệu) sẽ làm mất đi lượng lớn thông tin. Thay vào đó, có thể bỏ hoàn toàn cột này nếu không cần thiết, hoặc dùng các kỹ thuật impute phức tạp hơn nếu buộc phải giữ.

## D3. DataFrame sang NumPy

Dùng sáu cột trong `AUTO_FEATURES`. Drop các dòng thiếu ít nhất một trong
sáu cột, sau đó chuyển sang `float64` NumPy array và chuẩn hóa theo feature.

**Biến đầu ra bắt buộc**

- `analysis_df`
- `X_auto`
- `auto_feature_mean`
- `auto_feature_std`
- `X_auto_scaled`

In [ ]:
AUTO_FEATURES = ['curb_weight', 'engine_size', 'horsepower', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D2
df_clean = raw_df.replace('?', np.nan)
for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(df_clean[column])
missing_by_column = df_clean.isna().sum()

## D4. Outlier theo price z-score

Tính z-score của `price` bằng NumPy. Một dòng được xem là outlier trong bài
này khi `abs(z) > 2`.

**Biến đầu ra bắt buộc:** `price_z`, `price_outlier_mask`, `price_outliers`.

In [ ]:
# TODO D4
price_index = AUTO_FEATURES.index('price')
price_z = X_auto_scaled[:, price_index]
price_outlier_mask = np.abs(price_z) > 2
price_outliers = analysis_df[price_outlier_mask]

## D5. Correlation và GroupBy

**Biến đầu ra bắt buộc**

- `engine_price_corr`: Pearson correlation tính bằng NumPy.
- `price_by_body_style`: Series mean price theo `body_style`, sort index.

In [ ]:
# TODO D5
engine_index = AUTO_FEATURES.index('engine_size')
# Hàm np.corrcoef trả về ma trận tương quan 2x2, ta lấy phần tử ngoài đường chéo
engine_price_corr = np.corrcoef(X_auto[:, engine_index], X_auto[:, price_index])[0, 1]
price_by_body_style = df_clean.groupby('body_style')['price'].mean().sort_index()

# Phần 3. Visualization và insight

Mỗi biểu đồ cần:

1. một câu hỏi;
2. title, axis labels và unit;
3. lựa chọn chart phù hợp;
4. 1--2 câu nhận xét ngay dưới chart.

## M2.1 Price phân phối như thế nào?

In [ ]:
# TODO M2.1: histogram/KDE của price
plt.title("Phân phối giá xe (Price Distribution)")
sns.histplot(data=df_clean, x='price', kde=True)
plt.xlabel("Price (USD)")
plt.ylabel("Count")

**Nhận xét:** <!-- 1--2 câu -->
Phân phối giá xe lệch phải (right-skewed). Đa số các mẫu xe tập trung ở phân khúc giá rẻ và trung bình (dưới 15,000 USD), và rất ít xe ở phân khúc cao cấp (trên 30,000 USD).

## M2.2 Dataset có cân bằng theo body style không?

In [ ]:
# TODO M2.2: countplot của body_style
plt.title("Số lượng mẫu xe theo kiểu dáng (Body Style)")
sns.countplot(data=df_clean, x='body_style', order=df_clean['body_style'].value_counts().index)
plt.xlabel("Body Style")
plt.ylabel("Số lượng xe")

**Nhận xét:** <!-- 1--2 câu -->
Dữ liệu mất cân bằng nặng theo body_style. Sedan và hatchback chiếm đại đa số, trong khi các kiểu như convertible hay hardtop có lượng mẫu rất ít.

## M2.3 Price khác nhau theo body style ra sao?

In [ ]:
# TODO M2.3: boxplot price theo body_style
plt.title("Phân bố giá xe theo kiểu dáng")
sns.boxplot(data=df_clean, x='body_style', y='price')
plt.xlabel("Body Style")
plt.ylabel("Price (USD)")

**Nhận xét:** <!-- 1--2 câu -->
Xe mui trần (convertible) và hardtop có mức giá trung bình cao nhất và khoảng giá dao động rộng. Hatchback thường có giá thấp nhất và tập trung trong khoảng hẹp.

## M2.4 Engine size liên quan thế nào tới price?

In [ ]:
# TODO M2.4: scatterplot engine_size và price, hue=fuel_type
plt.title("Tương quan giữa Dung tích động cơ và Giá xe")
sns.scatterplot(data=df_clean, x='engine_size', y='price', hue='fuel_type')
plt.xlabel("Engine Size (cubic inches)")
plt.ylabel("Price (USD)")

**Nhận xét:** <!-- 1--2 câu -->
Có sự tương quan dương tuyến tính mạnh giữa dung tích động cơ và giá xe. Xe chạy bằng xăng (gas) chiếm đa số ở phân khúc dung tích và giá cao.

## M2.5 Các feature numeric tương quan ra sao?

In [ ]:
# TODO M2.5: correlation heatmap
plt.title("Correlation Heatmap của các biến số học")
sns.heatmap(df_clean[NUMERIC_COLUMNS].corr(), annot=False, cmap='coolwarm')

**Nhận xét:** <!-- 1--2 câu -->
engine_size, horsepower, curb_weight, và length có tương quan dương rất mạnh với price. Trái lại, city_mpg và highway_mpg (chỉ số tiết kiệm nhiên liệu) tương quan âm với price.

## M2.6 Biểu đồ tự chọn

Đặt một câu hỏi mới, chọn chart phù hợp và không lặp nguyên năm biểu đồ trên.

In [ ]:
# TODO M2.6: biểu đồ tự chọn
plt.title("Phân bố Mã lực theo Cơ chế nạp khí")
sns.boxplot(data=df_clean, x='aspiration', y='horsepower')
plt.xlabel("Aspiration")
plt.ylabel("Horsepower (hp)")

**Nhận xét:** <!-- 1--2 câu -->
Động cơ turbo (tăng áp) có mức mã lực trung bình cao hơn rõ rệt so với động cơ hút khí tự nhiên (std), tuy nhiên phổ dao động của std lại rộng hơn với nhiều điểm dị biệt (outlier) ở mức cao.

# Tổng hợp

Viết:

- 3--5 phát hiện chính có dẫn chứng;
- ít nhất 2 hạn chế của dataset;
- một ví dụ về correlation không đồng nghĩa causation;
- một câu hỏi nên phân tích tiếp.

## Tổng hợp của sinh viên

<!-- Viết khoảng 150--250 từ. -->
1. Các phát hiện chính (có dẫn chứng)
* Phân phối giá xe lệch phải rõ rệt: Phần lớn các mẫu xe trong bộ dữ liệu tập trung ở phân khúc phổ thông có mức giá từ 5,000 USD đến 15,000 USD. Chỉ có một số lượng rất nhỏ các dòng xe thuộc phân khúc cao cấp của các hãng như BMW, Mercedes-Benz, hay Jaguar có mức giá đột biến vượt trên 30,000 USD.  
* Tương quan thuận mạnh giữa dung tích động cơ và giá xe: Thuộc tính engine_size tỷ lệ thuận mạnh mẽ với price. Minh chứng là các mẫu xe đắt tiền như Jaguar hay Mercedes-Benz đều sở hữu dung tích động cơ lớn (từ 258 đến 308 cubic inches) , ngược lại các dòng xe giá rẻ như Chevrolet hay Honda có dung tích động cơ rất nhỏ (chỉ từ 61 đến 92 cubic inches).  
* Đánh đổi giữa hiệu suất nhiên liệu và sức mạnh động cơ: Có một sự tương quan nghịch rõ rệt giữa mã lực (horsepower) và chỉ số tiết kiệm nhiên liệu (city_mpg / highway_mpg). Các dòng xe thể thao hoặc xe sang có mã lực lớn như Porsche hay Jaguar có mức tiêu thụ nhiên liệu rất cao trong đô thị (chỉ đạt 13 - 17 mpg) , trong khi các dòng xe phổ thông như Chevrolet hay Honda lại tối ưu nhiên liệu rất tốt (đạt từ 38 - 49 mpg).  
2. Các hạn chế của dataset
* Tỷ lệ dữ liệu thiếu (Missing values) nghiêm trọng: Thuộc tính normalized_losses bị thiếu tới 41 giá trị (hiển thị dưới dạng ?), điều này làm giảm đáng kể độ tin cậy khi phân tích rủi ro bảo hiểm hoặc buộc người xử lý phải loại bỏ gần 20% lượng dữ liệu của bộ thuộc tính này. Ngoài ra, cột biến mục tiêu price cũng bị khuyết ở một số dòng xe.  
* Tính lỗi thời của dữ liệu: Bộ dữ liệu này được thu thập từ năm 1985 (1985 Auto Imports), vì vậy các thông số định lượng về giá cả (USD), công nghệ chế tạo động cơ, và tiêu chuẩn khí thải/tiết kiệm nhiên liệu không còn mang tính thời sự và không thể đại diện cho thị trường ô tô hiện đại ngày nay.
3. Ví dụ về "Correlation không đồng nghĩa với Causation"
* Ví dụ: Phân tích cho thấy có sự tương quan âm mạnh mẽ giữa mức tiết kiệm nhiên liệu (city_mpg) và giá xe (price) — tức là xe có số mpg càng cao thì giá bán lại càng rẻ. Tuy nhiên, khả năng tiết kiệm xăng tốt không phải là nguyên nhân khiến chiếc xe bị hạ giá. Mối quan hệ nhân quả thực sự ở đây bị chi phối bởi các yếu tố cốt lõi bên trong (biến ẩn): xe giá rẻ thường có kích thước nhỏ, trọng lượng nhẹ và sử dụng động cơ dung tích nhỏ nên tiêu thụ ít nhiên liệu; trong khi xe đắt tiền thường trang bị động cơ hiệu năng cao, thân vỏ nặng và nhiều tiện ích phức tạp khiến xe hao xăng hơn.  
4. Câu hỏi nên phân tích tiếp"
* Cơ chế nạp khí động cơ (aspiration gồm std và turbo) khi kết hợp với loại nhiên liệu sử dụng (fuel_type gồm gas và diesel) sẽ ảnh hưởng cụ thể như thế nào đến tốc độ đạt công suất cực đại (peak_rpm) và mức độ tổn thất bảo hiểm (normalized_losses) của các hãng xe?"